In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn

print(tf.__version__)
print(np.__version__)
print(sklearn.__version__)

In [ ]:
df = pd.read_csv('yoga_pose.csv')
df.head()

In [ ]:
# Hip midpoint should already be close to 0 because capture-time normalization is applied.
print(df[['x23', 'y23', 'z23', 'x24', 'y24', 'z24']].describe())

In [ ]:
# input into model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df.drop('label', axis=1).values

le = LabelEncoder()
y = le.fit_transform(df['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2026)

print(le.classes_)  # shows which index maps to which pose
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

In [ ]:
# input into model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model = Sequential()
model.add(tf.keras.Input(shape=(132, )))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(len(le.classes_), activation='softmax'))

model.compile(optimizer=Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Train the model
model.fit(X_train, y_train, batch_size=32, epochs=50, validation_split=0.1, callbacks=[stop_early])
# observe output of model on test set
y_conf= model.predict(X_test)
y_pred = np.argmax(y_conf, axis=1)

y_pred_cat = []
for pred in y_pred:
    y_pred_cat.append(le.classes_[pred])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
import json

model.save('yoga_pose_classifier.keras', overwrite=True)
config = {
    'label_encoder': le.classes_.tolist()
    }
with open('model_config.json', 'w') as f:
    json.dump(config, f)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('yoga_pose_classifier.tflite', 'wb') as f:
    f.write(tflite_model)